In [0]:
# Cell 1
%load_ext autoreload
%autoreload 2

import sys
import os
# 确保你的 src 目录在系统路径中，以便能够 import nyc_taxi_pipeline
sys.path.append(os.path.abspath("../src"))

In [0]:
%pip install geopandas
%pip install h3 
%pip install pandas 

In [0]:
 
%pip install -r requirements.txt

In [0]:
import os
import sys

# 假设你的 Notebook 在 notebooks/ 目录下
repo_root = "/Workspace/Repos/gloriatt502@gmail.com/nyc-taxi-pipeline"
src_path = os.path.join(repo_root, "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path) # 用 insert(0, ...) 确保最高优先级

# 打印一下 src 目录下的真实内容，看看文件夹名字对不对
print(f"src 目录下的内容: {os.listdir(src_path)}")

try:
    # 尝试列出 loaders 目录
    loaders_path = os.path.join(src_path, "nyc_taxi_pipeline", "loaders")
    print(f"loaders 目录下的内容: {os.listdir(loaders_path)}")
except Exception as e:
    print(f"无法访问子目录: {e}")

# 重新尝试导入
from nyc_taxi_pipeline.silver.nyc_taxi_silver import NYC_Taxi_Silver_Loader

In [0]:
import sys
import os
from unittest.mock import MagicMock
import traceback

# 1. 修复路径：请确保这里的路径指向你 git repo 的 src 目录
# 假设你的 Notebook 在 /Workspace/Users/.../notebooks
# 我们需要把 src 所在的绝对路径加进来
repo_root = os.path.abspath("..") # 根据你的实际结构调整，确保能看到 nyc_taxi_pipeline
src_path = os.path.join(repo_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

# 2. 定义一个“陷阱”类
class DatabricksSDKTrap(MagicMock):
    def __getattr__(self, name):
        # 当代码尝试访问任何属性（如 WorkspaceClient）时，触发堆栈打印
        print(f"\n🚨 警报！检测到对 Databricks SDK 属性的访问: {name}")
        traceback.print_stack()
        return super().__getattr__(name)

# 3. 注入陷阱（不直接实例化，而是作为一个可以被调用的 Mock）
mock_sdk = MagicMock()
# 让 WorkspaceClient 返回这个陷阱
mock_sdk.WorkspaceClient = DatabricksSDKTrap 
sys.modules['databricks.sdk'] = mock_sdk

print(f"当前搜索路径: {sys.path[-1]}")
print("开始尝试导入 Silver Loader...")

# 4. 尝试导入
try:
    from nyc_taxi_pipeline.loaders.silver_loader import NYC_Taxi_Silver_Loader
    print("\n✅ 导入成功！如果你没看到上面的 '🚨 警报'，说明导入阶段是安全的。")
except Exception as e:
    print(f"\n❌ 导入失败: {e}")

In [0]:
import sys
from unittest.mock import MagicMock

# 1. 定义一个特殊的 Mock 对象，当它被访问时打印堆栈
class TracebackMock(MagicMock):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        import traceback
        print("--- 检测到 Databricks SDK 初始化尝试！ ---")
        traceback.print_stack()
        print("---------------------------------------")

# 2. 注入 Mock
sys.modules['databricks.sdk'] = TracebackMock()

# 3. 尝试导入你的 Loader
try:
    from nyc_taxi_pipeline.loaders.silver_loader import NYC_Taxi_Silver_Loader
    print("\n✅ 导入逻辑执行完毕。")
except Exception as e:
    print(f"\n❌ 导入过程中发生错误: {e}")

In [0]:
import sys
import os

# 假设你的 Notebook 放在项目根目录的 notebooks/ 文件夹下
# 那么 src 文件夹相对路径就是 ../src
src_path = os.path.abspath("../src")

# 把 src 目录加入到 Python 的搜索路径中
if src_path not in sys.path:
    sys.path.append(src_path)
    print(f"已将路径加入 sys.path: {src_path}")

# 开启热更新（如果你修改了 .py 文件，Notebook 会自动加载新代码，不用重启）
%load_ext autoreload
%autoreload 2

# 现在可以正常 import 了！
from nyc_taxi_pipeline.spatial.build_zone_lookup import process_spatial_to_h3
print("成功导入模块！")

In [0]:
# Cell 2: 测试 Python 空间计算
import geopandas as gpd
from shapely.geometry import Polygon
from nyc_taxi_pipeline.spatial.build_zone_lookup import process_spatial_to_h3

# 1. 构造一个包含曼哈顿大概坐标的假 Polygon
polygon = Polygon([(-74.0, 40.7), (-74.0, 40.8), (-73.9, 40.8), (-73.9, 40.7)])

mock_gdf = gpd.GeoDataFrame({
    "LocationID": [1],
    "borough": ["Manhattan"],
    "zone": ["Test Zone"],
    "geometry": [polygon]
}, crs="EPSG:4326")

# 2. 调用你在 .py 里写的核心业务函数
result_pdf = process_spatial_to_h3(mock_gdf)

# 3. 直观地查看结果 (如果是 Databricks 环境，推荐使用 display())
display(result_pdf) 

# 输出结果应该包含：LocationID = 1, centroid_lat = 40.75, centroid_lng = -73.95, 以及生成的 h3_cell